# K-Means Project Tutorial: House grouping system

Classify California census-block groups by **region** and **median income** using `MedInc`, `Latitude`, and `Longitude`.

The train/test split is not used for supervised metrics here. We fit K-Means on `train`, then assign each unseen `test` house to the cluster it belongs to.

## Step 1: Loading the dataset

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

DATA_PATH = Path("../data/raw/housing.csv")

total_data = pd.read_csv(DATA_PATH)
print(f"Shape: {total_data.shape}")
print(f"Missing values: {total_data.isnull().sum().sum()}")
total_data.head()

### Keep only the clustering features

In [ ]:
features = ["MedInc", "Latitude", "Longitude"]
X = total_data[features].copy()

print(X.describe())
X.head()

### Split into train and test

K-Means will learn the cluster centers from `X_train`. `X_test` is held out so we can later predict which cluster new houses belong to.

In [ ]:
X_train, X_test = train_test_split(X, test_size=0.2, random_state=42)
X_train = X_train.copy()
X_test = X_test.copy()

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")
X_train.head()

## Step 2: Build a K-Means

Classify the training houses into **6 clusters**. Store the assigned group as a `cluster` column, inspect its format, and convert it to a category if the labels are discrete group IDs rather than a numeric scale.

In [ ]:
from sklearn.cluster import KMeans

model_unsup = KMeans(n_clusters=6, n_init="auto", random_state=42)
model_unsup.fit(X_train[features])

X_train["cluster"] = model_unsup.labels_

print("cluster dtype:", X_train["cluster"].dtype)
print("unique values:", sorted(X_train["cluster"].unique()))
print(X_train["cluster"].value_counts().sort_index())

# Labels are integers 0-5. They name groups, not a numeric scale.
X_train["cluster"] = X_train["cluster"].astype("category")
print("\ncluster dtype after categorizing:", X_train["cluster"].dtype)
X_train.head()

### Plot the training clusters

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axis = plt.subplots(1, 3, figsize=(15, 5))

sns.scatterplot(ax=axis[0], data=X_train, x="Longitude", y="Latitude", hue="cluster", palette="deep")
sns.scatterplot(ax=axis[1], data=X_train, x="Latitude", y="MedInc", hue="cluster", palette="deep")
sns.scatterplot(ax=axis[2], data=X_train, x="Longitude", y="MedInc", hue="cluster", palette="deep")

axis[0].set_title("Geographic clusters")
axis[1].set_title("Income vs latitude")
axis[2].set_title("Income vs longitude")
plt.tight_layout()
plt.show()

In [ ]:
cluster_profile = (
    X_train.groupby("cluster", observed=True)[["MedInc", "Latitude", "Longitude"]]
    .mean()
    .round(2)
)
cluster_profile["houses"] = X_train["cluster"].value_counts().sort_index()
cluster_profile

`cluster` starts as integers `0`–`5`. Those values only name groups, so the column is stored as a category before plotting.

The Longitude vs Latitude scatter splits California into two regions, then income splits each region:

- **Northern California** (higher latitude, more negative longitude): cluster **0** is upper-middle income (mean MedInc ≈ 5.4, including much of the Bay Area) and cluster **5** is a large lower-income northern group (mean ≈ 2.7).
- **Southern California** (Los Angeles, Orange County, San Diego): cluster **2** is affluent (mean ≈ 6.9), cluster **1** is middle income (mean ≈ 4.4), and cluster **3** is the large lower-income southern group (mean ≈ 2.4).
- **Cluster 4** is a small set of very high-income houses (n = 246, mean MedInc ≈ 11.8). It is not one city; those points sit in wealthy coastal pockets.

K-Means is grouping houses by **where they are** and **how wealthy the block is**, which matches the goal of classifying houses by region and median income.

## Step 3: Predict with the test set

The test houses were never seen during `fit`. Use the trained model to predict the cluster each one belongs to, then add those points to the plot above to confirm whether the prediction is successful.

In [ ]:
X_test["cluster"] = model_unsup.predict(X_test[features])
X_test["cluster"] = X_test["cluster"].astype("category")

print(X_test["cluster"].value_counts().sort_index())
X_test.head()

In [ ]:
comparison = pd.DataFrame({
    "train_%": (X_train["cluster"].value_counts(normalize=True).sort_index() * 100).round(2),
    "test_%": (X_test["cluster"].value_counts(normalize=True).sort_index() * 100).round(2),
    "test_houses": X_test["cluster"].value_counts().sort_index(),
    "train_MedInc": X_train.groupby("cluster", observed=True)["MedInc"].mean().round(2),
    "test_MedInc": X_test.groupby("cluster", observed=True)["MedInc"].mean().round(2),
})
comparison

### Overlay test predictions on the training plot

In [ ]:
centers = pd.DataFrame(model_unsup.cluster_centers_, columns=features)

fig, axis = plt.subplots(1, 3, figsize=(15, 5))

pairs = [("Longitude", "Latitude"), ("Latitude", "MedInc"), ("Longitude", "MedInc")]

for ax, (x_col, y_col) in zip(axis, pairs):
    # Training houses: faded background
    sns.scatterplot(ax=ax, data=X_train, x=x_col, y=y_col, hue="cluster", palette="deep", alpha=0.25, s=12, linewidth=0, legend=(ax is axis[0]))
    # Test predictions: "+" markers on top
    sns.scatterplot(ax=ax, data=X_test, x=x_col, y=y_col, hue="cluster", palette="deep", marker="+", s=22, linewidth=0.7, legend=False)
    # Cluster centers learned from train
    ax.scatter(centers[x_col], centers[y_col], c="black", marker="X", s=160, zorder=5)
    ax.set_title(f"{y_col} vs {x_col}")

axis[0].set_title("Test predictions over training clusters")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

# Distance from each house to the center of the cluster it was assigned to.
train_dist = np.linalg.norm(X_train[features].values - model_unsup.cluster_centers_[X_train["cluster"].astype(int)], axis=1)
test_dist = np.linalg.norm(X_test[features].values - model_unsup.cluster_centers_[X_test["cluster"].astype(int)], axis=1)

print(f"Mean distance to assigned center - train: {train_dist.mean():.3f} | test: {test_dist.mean():.3f}")

# Every test house should be closest to the center it was assigned to.
nearest = np.argmin(np.linalg.norm(X_test[features].values[:, None, :] - model_unsup.cluster_centers_[None, :, :], axis=2), axis=1)
print(f"Test houses assigned to their nearest center: {(nearest == X_test['cluster'].astype(int)).mean():.1%}")

**The prediction is successful.** The `+` markers (test houses) fall inside the same colored regions as the faded training points, with no stray markers landing in a foreign group. The black `X` markers are the centers learned from `train`, and the test points sit around those same centers.

The numbers confirm what the plot shows:

- **Cluster sizes match.** Each cluster holds nearly the same share of houses in both sets (for example cluster 3: 26.45% of train vs 26.91% of test; cluster 4: 1.49% vs 1.50%).
- **Cluster meaning is stable.** Mean income per cluster is almost identical across sets (cluster 2: 6.94 train vs 6.96 test; cluster 4: 11.75 vs 11.72).
- **Test points are not worse-fitted.** Mean distance to the assigned center is 1.217 for train and 1.200 for test, so unseen houses sit just as close to their group as the houses used for fitting.
- **Every test house was assigned to its nearest center** (100% agreement), which is exactly what `predict` should do.

Since the held-out houses land in the same regional and income groups as the training data, the model generalizes to new points rather than memorizing the training set.

In [ ]:
from pathlib import Path
from pickle import dump

processed_dir = Path("../data/processed")
models_dir = Path("../models")
processed_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)

X_train.to_csv(processed_dir / "housing_train.csv", index=False)
X_test.to_csv(processed_dir / "housing_test.csv", index=False)

with open(models_dir / "kmeans_housing.sav", "wb") as file:
    dump(model_unsup, file)

print("Saved train/test CSVs and the K-Means model.")